# Домашнее задание №5 — Контроль качества и тримминг (FastQC / fastp / MultiQC)

**Датасет:** PRJEB84057 (Illumina HiSeq 4000, single-end)

**Образцы (4 шт):**
- ERR14230582
- ERR14230586
- ERR14230595
- ERR14230607



## Часть 1 — Анализ FastQC (до тримминга)

### Проблемные аспекты, выявленные в MultiQC-отчёте `multiqc_before.html`

| Метрика | Состояние | Вывод |
|---|---|---|
| Per Base Sequence Quality | Зелёная зона, провалов Q<20 в хвостах нет | Качество ридов высокое |
| Adapter Content | < 0.1% контаминации во всех 4 образцах (скрыты по умолчанию в MultiQC) | Остаточные адаптеры есть — удалим auto-detection |
| Per Base Sequence Content | Стабильное распределение A/T/G/C | Без особенностей |
| Overrepresented sequences | Нет частых последовательностей | Чисто |
| Per Sequence Quality Scores | Пик на Q≈36, длинный хвост ОК | Высокое качество |

**Итог:** данные чистые. Адаптеры присутствуют в следовых количествах, скользящее окно нужно как страховка.

### Скрипт `02_fastqc_before.sh`

In [1]:
!cat scripts/02_fastqc_before.sh

#!/bin/bash
#SBATCH --job-name=fastqc_before
#SBATCH --output=../logs/fastqc_before_%j.out
#SBATCH --error=../logs/fastqc_before_%j.err
#SBATCH --cpus-per-task=4
#SBATCH --mem=4G
#SBATCH --time=01:00:00

DATA_DIR=../data
OUT_DIR=../results/fastqc_before
MULTIQC_DIR=../results/multiqc_before

mkdir -p $OUT_DIR $MULTIQC_DIR

fastqc -t 4 -o $OUT_DIR $DATA_DIR/*.fastq.gz

multiqc $OUT_DIR -o $MULTIQC_DIR -n multiqc_before


## Часть 2 — Тримминг (fastp)

Применены три обработки:
1. **Auto-detection адаптеров** (`--detect_adapter_for_pe` для совместимости).
2. **Скользящее окно 5:20** (`--cut_right --cut_right_window_size 5 --cut_right_mean_quality 20`) — обрезает с 3'-конца, если средний Q в окне < 20.
3. **Отбрасывание коротких ридов** (`--length_required 36`).

### Скрипт `03_fastp_trim.sh`

In [2]:
!cat scripts/03_fastp_trim.sh

#!/bin/bash
#SBATCH --job-name=fastp_trim
#SBATCH --output=../logs/fastp_%j.out
#SBATCH --error=../logs/fastp_%j.err
#SBATCH --cpus-per-task=4
#SBATCH --mem=4G
#SBATCH --time=01:00:00

DATA_DIR=../data
TRIM_DIR=../results/trimmed

mkdir -p $TRIM_DIR

for fq in $DATA_DIR/*.fastq.gz; do
    base=$(basename $fq .fastq.gz)
    fastp \
        -i $fq \
        -o $TRIM_DIR/${base}_trimmed.fastq.gz \
        --detect_adapter_for_pe \
        --cut_right \
        --cut_right_window_size 5 \
        --cut_right_mean_quality 20 \
        --length_required 36 \
        -j $TRIM_DIR/${base}_fastp.json \
        -h $TRIM_DIR/${base}_fastp.html \
        -w 4
done


### Папка с очищенными файлами

In [3]:
!ls -la results/trimmed/

total 175880
drwxr-xr-x  14 abbiasov91  staff       448 May 26 10:35 .
drwxr-xr-x   7 abbiasov91  staff       224 May 26 10:11 ..
-rw-r--r--   1 abbiasov91  staff    221636 May 26 10:35 ERR14230582_Illumina_HiSeq_4000_sequencing_fastp.html
-rw-r--r--   1 abbiasov91  staff     48578 May 26 10:35 ERR14230582_Illumina_HiSeq_4000_sequencing_fastp.json
-rw-r--r--   1 abbiasov91  staff   9592833 May 26 10:35 ERR14230582_Illumina_HiSeq_4000_sequencing_trimmed.fastq.gz
-rw-r--r--   1 abbiasov91  staff    222677 May 26 10:35 ERR14230586_Illumina_HiSeq_4000_sequencing_fastp.html
-rw-r--r--   1 abbiasov91  staff     50741 May 26 10:35 ERR14230586_Illumina_HiSeq_4000_sequencing_fastp.json
-rw-r--r--   1 abbiasov91  staff  43495797 May 26 10:35 ERR14230586_Illumina_HiSeq_4000_sequencing_trimmed.fastq.gz
-rw-r--r--   1 abbiasov91  staff    222280 May 26 10:35 ERR14230595_Illumina_HiSeq_4000_sequencing_fastp.html
-rw-r--r--   1 abbiasov91  staff     49258 May 26 10:35 ERR14230595_Illumina_HiSeq_4000_

## Часть 3 — Контроль качества после тримминга

### Скрипт `04_fastqc_after.sh`

In [4]:
!cat scripts/04_fastqc_after.sh

#!/bin/bash
#SBATCH --job-name=fastqc_after
#SBATCH --output=../logs/fastqc_after_%j.out
#SBATCH --error=../logs/fastqc_after_%j.err
#SBATCH --cpus-per-task=4
#SBATCH --mem=4G
#SBATCH --time=01:00:00

TRIM_DIR=../results/trimmed
OUT_DIR=../results/fastqc_after
MULTIQC_DIR=../results/multiqc_after

mkdir -p $OUT_DIR $MULTIQC_DIR

fastqc -t 4 -o $OUT_DIR $TRIM_DIR/*_trimmed.fastq.gz

multiqc $OUT_DIR $TRIM_DIR -o $MULTIQC_DIR -n multiqc_after


### Сравнение метрик до и после тримминга

Извлечём ключевые показатели из JSON-отчётов fastp:

In [5]:
import json, glob, os
import pandas as pd

rows = []
for j in sorted(glob.glob('results/trimmed/*_fastp.json')):
    d = json.load(open(j))
    name = os.path.basename(j).replace('_Illumina_HiSeq_4000_sequencing_fastp.json', '')
    br = d['summary']['before_filtering']
    af = d['summary']['after_filtering']
    filt = d['filtering_result']
    rows.append({
        'Образец': name,
        'Reads до': br['total_reads'],
        'Reads после': af['total_reads'],
        'Q20 до, %': round(br['q20_rate']*100, 2),
        'Q20 после, %': round(af['q20_rate']*100, 2),
        'Q30 до, %': round(br['q30_rate']*100, 2),
        'Q30 после, %': round(af['q30_rate']*100, 2),
        'Ср. длина до': br['read1_mean_length'],
        'Ср. длина после': af['read1_mean_length'],
        'Отброшено low-qual': filt['low_quality_reads'],
        'Отброшено too-short': filt['too_short_reads'],
    })

df = pd.DataFrame(rows)
df

,Образец,Reads до,Reads после,"Q20 до, %","Q20 после, %","Q30 до, %","Q30 после, %",Ср. длина до,Ср. длина после,Отброшено low-qual,Отброшено too-short
0,ERR14230582,576623,396126,99.60,99.69,98.81,98.96,40,44,4,180493
1,ERR14230586,1826305,1450141,99.68,99.77,99.00,99.17,44,47,30,376134
2,ERR14230595,1128748,791927,99.61,99.72,98.89,99.05,40,44,13,336808
3,ERR14230607,1054411,719112,99.60,99.71,98.87,99.04,40,44,12,335287


### Что улучшилось

| Метрика | До | После | Комментарий |
|---|---|---|---|
| Q20-rate | ~99.60% | ~99.71% | Скользящее окно срезало низкокачественные хвосты |
| Q30-rate | ~98.85% | ~99.05% | Доля высокоточных оснований выросла |
| Adapter content | <0.1% (зелёная зона) | 0% (полностью убраны) | Auto-detection fastp удалил остатки Illumina-адаптеров |
| Overrepresented sequences | Не было | Не появилось | OK |

### Почему это произошло
- **`--cut_right 5:20`** идёт от 3'-конца окнами по 5 п.н. и обрезает рид, как только среднее качество в окне падает < Q20. Это убирает «угасание» сигнала к концу ридов, типичное для Illumina.
- **`--detect_adapter_for_pe`** (даже для SE) включает поиск Illumina TruSeq/Nextera через overlap-анализ и удаляет их.
- **`--length_required 36`** отсекает риды, ставшие слишком короткими после обрезки — короткие фрагменты дают неуверенное выравнивание и шумят.

## Выводы

1. Сырые данные PRJEB84057 уже были высокого качества (Q20 ≈ 99.6%), но MultiQC выявил **остаточные Illumina-адаптеры < 0.1%** и риски качества к 3'-концу.
2. Тримминг fastp с параметрами `auto-detect adapters / cut_right 5:20 / min_len 36` улучшил Q20 и Q30 на всех 4 образцах, полностью устранил адаптеры.